# Demo for Section 4 - optim, utils and pipeline

Steps:
- optim: finite-difference gradients and Jacobians on scalar and vector functions, choosing the step `h`, and a tiny model that learns using `grad`
- utils: distances, nearest neighbours, stable activations (sigmoid, softmax, logsumexp) and batching
- pipeline: chaining the preprocessing steps from Section 1, adding a model at the end, and what mistakes look like
- benchmarks: Python loops vs NumPy, run from `benchmarks/section4_benchmarks.py`

`sample_data.csv` should be in the `demo` folder. A lowercase `x` is one point (a short list of numbers), a capital `X` is the whole data table.
The numbers in the saved outputs come from the computer that ran this notebook, so run all cells again to refresh them on yours.

In [2]:
import sys, platform, pathlib
import numpy as np

# to import numcompute from demo folder
ROOT = pathlib.Path.cwd()
if not (ROOT / "numcompute").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from numcompute import utils
from numcompute.io import load_csv
from numcompute.optim import grad, jacobian
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler, SimpleImputer, OneHotEncoder, NotFittedError

DATA = ROOT / "demo" / "sample_data.csv"
np.set_printoptions(precision=4, suppress=True)
print("NumPy", np.__version__, "| Python", platform.python_version())

NumPy 1.26.2 | Python 3.12.0


# 1. Gradients and Jacobians (optim.py)

A gradient is a list of slopes, one for each input of a function. `grad` measures it by nudging each input a tiny step `h` and seeing how much the output moves. `forward` only steps ahead, `central` steps both ways, so it is more accurate.
Here `f` has two inputs and we measure the slope at the point (2, 1). From calculus the exact answer is (2*x0, 3) = (4, 3).

In [3]:
f = lambda v: v[0] ** 2 + 3 * v[1]      # a function of two inputs
x = np.array([2.0, 1.0])                 # the point where we measure the slope
exact = np.array([2 * x[0], 3.0])

with np.printoptions(precision=8):       # show more digits so the small errors are visible
    print("exact      :", exact)
    print("forward    :", grad(f, x, method="forward"))
    print("central    :", grad(f, x))
print("x unchanged:", x)

exact      : [4. 3.]
forward    : [4.00001 3.     ]
central    : [4. 3.]
x unchanged: [2. 1.]


Choosing `h` is a balance. If `h` is too big the estimate is crude, if it is too small the computer's rounding errors take over. The table shows the error for the slope of sin at x = 1 (the true slope is cos 1).

In [4]:
sin_fn = lambda v: np.sin(v[0])
print(f"{'h':>8} | {'forward error':>14} | {'central error':>14}")
for h in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11, 1e-13]:
    err_forward = abs(grad(sin_fn, [1.0], h=h, method="forward")[0] - np.cos(1.0))
    err_central = abs(grad(sin_fn, [1.0], h=h)[0] - np.cos(1.0))
    print(f"{h:8.0e} | {err_forward:14.2e} | {err_central:14.2e}")

       h |  forward error |  central error
   1e-01 |       4.29e-02 |       9.00e-04
   1e-03 |       4.21e-04 |       9.01e-08
   1e-05 |       4.21e-06 |       1.17e-11
   1e-07 |       4.21e-08 |       2.10e-10
   1e-09 |       7.84e-09 |       1.77e-08
   1e-11 |       1.21e-06 |       1.21e-06
   1e-13 |       3.02e-04 |       4.69e-05


The Jacobian is the same idea for a function with several outputs. It is a table where row i, column j says how much output i changes when input j is nudged. This `F` has 2 inputs and 4 outputs, so the table is 4 x 2.

In [5]:
F = lambda v: np.array([v[0] * v[1], v[0] + v[1], v[1] ** 2, np.sin(v[0])])
p = np.array([1.0, 2.0])

J = jacobian(F, p)
J_exact = np.array([[p[1], p[0]], [1, 1], [0, 2 * p[1]], [np.cos(p[0]), 0]])
print("shape:", J.shape, "(4 outputs x 2 inputs)")
print(J)
print("biggest difference from the exact table:", float(np.max(np.abs(J - J_exact))))

shape: (4, 2) (4 outputs x 2 inputs)
[[2.     1.    ]
 [1.     1.    ]
 [0.     4.    ]
 [0.5403 0.    ]]
biggest difference from the exact table: 1.1681211553593585e-11


Why gradients matter: a model can learn by walking downhill. Here a straight line (slope and intercept) is fitted to age and income from the sample file, using only the complete rows. The loss is the average squared error, and `grad` tells us how to change the slope and intercept to make it smaller. Both columns are standardised first so the steps are well behaved.

In [6]:
NUM = dict(skip_header=1, usecols=(0, 1, 2), missing_values="NA")
X = load_csv(DATA, **NUM)                              # 12 rows, NaN where a value is missing
complete = load_csv(DATA, nan_policy="omit", **NUM)    # only the rows with no missing value
age, income = complete[:, 0], complete[:, 1]
a = (age - age.mean()) / age.std()
inc = (income - income.mean()) / income.std()

def loss(w):                                           # w = [slope, intercept]
    return np.mean((w[0] * a + w[1] - inc) ** 2)

w = np.array([0.0, 0.0])
print(f"{'step':>5} | {'slope':>8} {'intercept':>10} | {'loss':>8}")
print(f"{0:5d} | {w[0]:8.4f} {w[1]:10.4f} | {loss(w):8.4f}")
for step in range(1, 201):
    w = w - 0.1 * grad(loss, w)                        # move against the slope
    if step in (1, 5, 20, 100, 200):
        print(f"{step:5d} | {w[0]:8.4f} {w[1]:10.4f} | {loss(w):8.4f}")

print("rows used:", len(complete))
print("best possible slope (the correlation):", round(float(np.corrcoef(a, inc)[0, 1]), 4))

 step |    slope  intercept |     loss
    0 |   0.0000     0.0000 |   1.0000
    1 |   0.1937     0.0000 |   0.6624
    5 |   0.6511     0.0000 |   0.1629
   20 |   0.9572    -0.0000 |   0.0624
  100 |   0.9684     0.0000 |   0.0622
  200 |   0.9684     0.0000 |   0.0622
rows used: 8
best possible slope (the correlation): 0.9684


Bad input gives a clear message instead of a confusing crash. For very large inputs the step can disappear in rounding (x + h == x), which is also reported.

In [7]:
def show_error(label, action):
    try:
        action()
    except (TypeError, ValueError) as err:
        print(f"{label:<22} {type(err).__name__}: {err}")

show_error("bad method", lambda: grad(f, x, method="sideways"))
show_error("h is zero", lambda: grad(f, x, h=0))
show_error("x is a table", lambda: grad(f, np.ones((2, 2))))
show_error("f returns 2 numbers", lambda: grad(lambda v: v * 2, x))
show_error("step lost in rounding", lambda: grad(lambda v: v[0] ** 2, [1e12]))
print("with a bigger h it works:", grad(lambda v: v[0] ** 2, [1e12], h=1e3))

bad method             ValueError: method must be 'central' or 'forward', got 'sideways'
h is zero              ValueError: h must be a positive finite number, got 0
x is a table           ValueError: x must be a non-empty 1D array, got shape (2, 2)
f returns 2 numbers    ValueError: f must return a single number, got shape (2,)
step lost in rounding  ValueError: h=1e-05 is too small for x[0]=1000000000000.0, x + h rounds back to x. Try a bigger h.
with a bigger h it works: [2.e+12]


# 2. Distances, activations and batching (utils.py)

First we get a clean numeric matrix from the sample file: fill the missing numbers with the column mean, then standardise, so every column counts equally. `Z` has one row for each of the 12 people.

In [8]:
X_imp = SimpleImputer("mean").fit_transform(X)
Z = StandardScaler().fit_transform(X_imp)
print("Z shape:", Z.shape)

print("distance from person 0 to person 1 :", utils.euclidean_distance(Z[0], Z[1]))
print("manhattan distance, same two people:", utils.manhattan_distance(Z[0], Z[1]))

# broadcasting gives the distance between every pair of people at once
D = utils.euclidean_distance(Z[:, None, :], Z[None, :, :])
print("distance matrix shape:", D.shape, "| symmetric:", bool(np.allclose(D, D.T)),
      "| zero diagonal:", bool(np.all(np.diag(D) == 0)))

Z shape: (12, 3)
distance from person 0 to person 1 : 1.728698684473228
manhattan distance, same two people: 2.824529277562169
distance matrix shape: (12, 12) | symmetric: True | zero diagonal: True


`k_nearest` finds the rows that are closest to a query. Person 0 is always its own nearest neighbour (distance 0), so we ask for 4 and read the other 3.

In [9]:
idx, dist = utils.k_nearest(Z, Z[:1], k=4)
print("closest rows to person 0:", idx[0])
print("their distances         :", dist[0])
print("person 0 ->", X[0], "| closest other person", idx[0][1], "->", X[idx[0][1]])

closest rows to person 0: [0 8 3 5]
their distances         : [0.     0.8306 1.0463 1.1949]
person 0 -> [   23. 42000.    38.] | closest other person 8 -> [   26. 36000.    35.]


The activation functions are written so that huge numbers do not break them. The normal formula for softmax turns `exp(1000)` into infinity, and the result becomes NaN. Subtracting the biggest value first keeps every number small without changing the answer.

In [10]:
scores = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):                        # hide numpy's overflow warnings for the naive version
    naive_softmax = np.exp(scores) / np.exp(scores).sum()
    naive_lse = np.log(np.exp(scores).sum())

print("naive softmax    :", naive_softmax)
print("stable softmax   :", utils.softmax(scores), "| adds up to", float(utils.softmax(scores).sum()))
print("naive logsumexp  :", naive_lse)
print("stable logsumexp :", float(utils.logsumexp(scores)), "| exact:", 1002 + np.log(1 + np.exp(-1) + np.exp(-2)))
print()
print("sigmoid([-1000, 0, 1000]):", utils.sigmoid([-1000.0, 0.0, 1000.0]))
print("relu([-2, 0, 3])         :", utils.relu([-2.0, 0.0, 3.0]))

naive softmax    : [nan nan nan]
stable softmax   : [0.09   0.2447 0.6652] | adds up to 0.9999999999999999
naive logsumexp  : inf
stable logsumexp : 1002.4076059644444 | exact: 1002.4076059644444

sigmoid([-1000, 0, 1000]): [0.  0.5 1. ]
relu([-2, 0, 3])         : [0. 0. 3.]


`batch_indices` splits the row numbers into groups. Shuffling needs a seed so the order is the same every run. The last batch can be smaller.

In [11]:
plain = utils.batch_indices(len(Z), 5)
mixed = utils.batch_indices(len(Z), 5, shuffle=True, seed=42)
mixed_again = utils.batch_indices(len(Z), 5, shuffle=True, seed=42)

print("no shuffle   :", [b.tolist() for b in plain])
print("shuffle (42) :", [b.tolist() for b in mixed])
print("same seed again gives the same batches:", all(np.array_equal(a, b) for a, b in zip(mixed, mixed_again)))
print("every row is in exactly one batch     :", sorted(np.concatenate(mixed).tolist()) == list(range(len(Z))))
print("first batch of the data:")
print(Z[plain[0]])

no shuffle   : [[0, 1, 2, 3, 4], [5, 6, 7, 8, 9], [10, 11]]
shuffle (42) : [[0, 7, 6, 9, 11], [3, 5, 2, 4, 10], [1, 8]]
same seed again gives the same batches: True
every row is in exactly one batch     : True
first batch of the data:
[[-1.4441 -1.213  -0.864 ]
 [-0.3078  0.     -0.3888]
 [ 0.2604  0.1649  0.7992]
 [-0.8759 -1.372   0.    ]
 [ 1.302   1.3837  1.9873]]


# 3. Chaining the steps (pipeline.py)

A Pipeline runs the steps one after another, and each step passes its output to the next one. Here the missing numbers are filled and then the columns are standardised, the same two steps we did by hand above. It learns from the data in `fit` and reuses what it learned later.

In [12]:
pipe = Pipeline([("fill", SimpleImputer("mean")), ("scale", StandardScaler())])
out = pipe.fit_transform(X)
by_hand = StandardScaler().fit_transform(SimpleImputer("mean").fit_transform(X))

print("steps:", [name for name, step in pipe.steps])
print("output shape:", out.shape, "| NaN left:", int(np.isnan(out).sum()))
print("same as running the steps by hand:", bool(np.allclose(out, by_hand)))
print("original X untouched (still has NaN):", bool(np.isnan(X).any()))

steps: ['fill', 'scale']
output shape: (12, 3) | NaN left: 0
same as running the steps by hand: True
original X untouched (still has NaN): True


A new person arrives with a missing income. `transform` does not learn anything again, it reuses the means and spreads that were learned from the training data.

In [13]:
new_person = np.array([[30.0, np.nan, 40.0]])          # age 30, income missing, hours 40
print("new person through the fitted pipeline:", pipe.transform(new_person))
print("income was filled with the training mean:", round(float(pipe.named_steps["fill"].statistics_[1]), 3))

new person through the fitted pipeline: [[-0.7812  0.     -0.3888]]
income was filled with the training mean: 64888.889


The text columns can go through a pipeline too, and the two results can be joined side by side to make the final feature matrix.

In [14]:
CAT = dict(skip_header=1, usecols=(3, 4), dtype=str, nan_policy="fill", fill_value="unknown")
C = load_csv(DATA, **CAT)

text_pipe = Pipeline([("encode", OneHotEncoder())])
H = text_pipe.fit_transform(C)
print("text columns:", C.shape, "->", H.shape)

final = np.hstack([pipe.transform(X), H])
print("final feature matrix:", final.shape, "= 3 numeric columns +", H.shape[1], "one-hot columns")

text columns: (12, 2) -> (12, 9)
final feature matrix: (12, 12) = 3 numeric columns + 9 one-hot columns


The last step can be a model. The model gets the answers `y` in `fit`, the earlier steps never see `y`. `predict` sends new data through the earlier steps and then asks the model. No real model is needed for this project, so this one is a tiny stand-in that always predicts the average of `y`. Here we predict income from age and hours.

In [15]:
class MeanModel:
    def fit(self, X, y=None):
        self.mean_ = float(np.mean(y))
        return self

    def predict(self, X):
        return np.full(len(X), self.mean_)

X_train, y_train = complete[:, [0, 2]], complete[:, 1]   # age and hours -> income
model_pipe = Pipeline([("scale", StandardScaler()), ("model", MeanModel())])
model_pipe.fit(X_train, y_train)

print("rows used:", len(X_train), "| the model learned the average income:", round(model_pipe.named_steps["model"].mean_, 1))
print("prediction for 2 new people:", model_pipe.predict(np.array([[30.0, 40.0], [55.0, 48.0]])))

rows used: 8 | the model learned the average income: 68125.0
prediction for 2 new people: [68125. 68125.]


Common mistakes and what they give. The checks that matter most (an empty list, a model in the middle, X and y of different sizes) have their own clear messages.

In [16]:
mistakes = {
    "empty steps": lambda: Pipeline([]),
    "forgot the brackets": lambda: Pipeline([("scale", StandardScaler)]).fit(X_imp),
    "model in the middle": lambda: Pipeline([("model", MeanModel()), ("scale", StandardScaler())]),
    "transform before fit": lambda: Pipeline([("scale", StandardScaler())]).transform(X_imp),
    "predict on a transformer": lambda: Pipeline([("scale", StandardScaler())]).fit(X_imp).predict(X_imp),
    "X and y different sizes": lambda: Pipeline([("model", MeanModel())]).fit(X_imp, np.array([1.0, 2.0])),
}
for label, action in mistakes.items():
    try:
        action()
    except (TypeError, ValueError, AttributeError, NotFittedError) as err:
        print(f"{label:<26} {type(err).__name__}: {err}")

empty steps                ValueError: steps must not be empty
forgot the brackets        TypeError: StandardScaler.fit() missing 1 required positional argument: 'X'
model in the middle        TypeError: step 'model' is not the last step, so it needs a transform method
transform before fit       NotFittedError: StandardScaler is not fitted yet; call fit(X) before transform(X)
predict on a transformer   AttributeError: 'StandardScaler' object has no attribute 'predict'
X and y different sizes    ValueError: X and y must have the same number of rows, got 12 and 2


# 4. Python loops vs vectorised code (benchmarks/section4_benchmarks.py)

In table A it is a plain Python loop against the numcompute function, with the loop getting plain lists.

In [18]:
sys.path.insert(0, str(ROOT / "benchmark"))
import section4_benchmarks as bench

rows_a = bench.run_benchmarks(n_points=200, n_values=20000, n_inputs=100, n_rows=5000, repeats=5, seed=0)

Section 4: optim, utils and pipeline | seed=0 | repeats=5
Sizes: n_points=200 (d=8, 100 queries, k=5) | n_values=20000 | n_inputs=100 (jacobian 33x50) | n_rows=5000
Timing: best of repeats after one warm-up; same repeats for both versions.
Loop versions get plain Python lists, converted before timing.

A. Python loop vs NumPy (numcompute)
Component                                        Loop (s)    NumPy (s)     Speed-up
------------------------------------------------------------------------------------
utils: euclidean distance matrix                 0.097057     0.002851       34.04x
utils: manhattan distance matrix                 0.046355     0.002744       16.89x
utils: k_nearest                                 0.052144     0.003133       16.64x
utils: sigmoid                                   0.004207     0.000232       18.15x
utils: relu                                      0.001309     0.000017       77.89x
utils: softmax (rows of 10)                      0.005142     0.000281

Reading the tables: the distance, softmax, sigmoid and Jacobian rows gain a lot, because the loop version does the arithmetic one number at a time in Python. `grad` gains much less at this size, because it has to call `f` once for every nudge whichever way it is written, so its time is set by the number of calls. It pulls ahead when `f` itself is expensive (try `n_inputs=1000`).